# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashritha-boop/fly_machine/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Source: [“Which Pages Should You Review First? An Early-Warning Queue for Organic Click Decline”](../../docs/paper-examples/index.html), the FlyRank worked-example paper. I read its two evaluations as bounded evidence rather than a verdict on the approach:

1. **Client-held-out development evaluation:** the target is a next-month click decline (next-month clicks below 80% of the prior month; observed base rate 0.416). Across five client-grouped folds, HistGradientBoosting measured precision@50 of 0.504 ± 0.211, compared with 0.384 ± 0.182 for the frozen rule. The paper also reports the per-fold spread and notes that the model was higher in three folds and lower in two. **Methodology question:** could the paper clarify how the 80% click threshold behaves for low-volume pages, and whether all included rows have complete adjacent-month coverage? The client grouping is a thoughtful match for cross-client transfer; with 36 clients and substantial fold variation, I would read the result as directional evidence for this evaluation population, not a guarantee for every portfolio.

2. **Shifted June evaluation:** the sealed-month base rate rose to 0.655; model precision@50 was 0.740, while the simple already-dipping list measured 0.780 and the frozen rule 0.380. **Methodology question:** could future reporting keep the model beside both the month-specific base rate and the simple list, and repeat this time-forward check across additional months? That would help distinguish useful ranking signal from a single shifted month's label prevalence. The paper appropriately treats June as one sealed month and describes monitoring as proposed policy, not deployed impact.

These are questions about label construction and the scope of validation, not about whether the work is worthwhile. Paper target: next-month **clicks**; my Week-5 target below is next-month **impressions**, so their metrics are not directly comparable.

In [1]:
paper_evidence = {
    "development label": "next-month clicks < 80% of prior-month clicks",
    "development base rate": 0.416,
    "development population": "16,513 eligible pages; 36 clients",
    "precision@50, client-fold mean": {
        "frozen rule": "0.384 +/- 0.182",
        "HistGradientBoosting": "0.504 +/- 0.211",
    },
    "shifted June check": {
        "base rate": 0.655,
        "frozen rule precision@50": 0.380,
        "model precision@50": 0.740,
        "already-dipping list precision@50": 0.780,
    },
}
for section, evidence in paper_evidence.items():
    print(f"{section}: {evidence}")
print("Paper numbers describe separate evaluations; neither target nor metric is my Week-5 result.")

development label: next-month clicks < 80% of prior-month clicks
development base rate: 0.416
development population: 16,513 eligible pages; 36 clients
precision@50, client-fold mean: {'frozen rule': '0.384 +/- 0.182', 'HistGradientBoosting': '0.504 +/- 0.211'}
shifted June check: {'base rate': 0.655, 'frozen rule precision@50': 0.38, 'model precision@50': 0.74, 'already-dipping list precision@50': 0.78}
Paper numbers describe separate evaluations; neither target nor metric is my Week-5 result.


## 2. My model under an honest split (before/after)

Week 5 predicts a specific observed proxy: April 2026 impressions are below 80% of March impressions. March measurements and content age as of March 31 are inputs; April impressions are used only to define the label. The paired-month frame is restricted to pages with at least 10 March impressions and data in both months.

For a before/after diagnostic, I rerun the same Week-5 model candidates and unchanged Week-4 rule on this same frame twice: five random-row 80/20 splits first, then five client-grouped folds. The random split is an intentionally weaker reference because the same clients can occur in train and test; client-held-out folds are the primary result for transfer to unseen clients. Both designs report the base rate and precision@50 on their own test rows. Fold-to-fold SD is descriptive, not a confidence interval. The highest-P@50 model is selected on these same grouped folds, so its score is exploratory and may be optimistic; this is not an untouched holdout or a time-forward evaluation.

In [2]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        "This notebook reruns the Week-5 warehouse model. Add the read-only HF_TOKEN "
        "to Colab Secrets after accepting the dataset terms; no synthetic fallback is used."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Match Week 5: March is the feature window; April is label-only.
base_sql = f"""
WITH march AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_march,
        SUM(gsc_clicks) AS gsc_clicks_march,
        AVG(gsc_avg_position) AS gsc_avg_position_march,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)
            AS ga4_sessions_march,
        CASE WHEN BOOL_OR(ga4_data_available IS TRUE) THEN 1 ELSE 0 END
            AS ga4_available_march
    FROM {FACT}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
),
april AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_april
    FROM {FACT}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.gsc_impressions_march,
    m.gsc_clicks_march,
    m.gsc_avg_position_march,
    m.ga4_sessions_march,
    m.ga4_available_march,
    DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
    a.gsc_impressions_april,
    CASE WHEN a.gsc_impressions_april < 0.80 * m.gsc_impressions_march
         THEN 1 ELSE 0 END AS is_declining_label
FROM march m
JOIN april a
  ON m.content_hash_id = a.content_hash_id
 AND m.client_hash_id = a.client_hash_id
JOIN {CONTENT} c
  ON m.content_hash_id = c.content_hash_id
 AND m.client_hash_id = c.client_hash_id
WHERE m.gsc_impressions_march >= 10
"""
df_data = con.sql(base_sql).df()
df_data["ga4_sessions_march"] = df_data["ga4_sessions_march"].where(
    df_data["ga4_available_march"].eq(1)
)
df_data = df_data.dropna(
    subset=[
        "gsc_impressions_march",
        "gsc_clicks_march",
        "gsc_avg_position_march",
        "content_age_days",
        "is_declining_label",
    ]
).reset_index(drop=True)
assert not df_data.duplicated(["client_hash_id", "content_hash_id"]).any()
assert df_data["is_declining_label"].isin([0, 1]).all()
assert df_data["client_hash_id"].nunique() >= 5

FEATURES = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "gsc_avg_position_march",
    "ga4_sessions_march",
    "content_age_days",
    "ga4_available_march",
    "position_available_march",
]
df_data["position_available_march"] = (
    df_data["gsc_avg_position_march"].gt(0).astype(int)
)
FORBIDDEN_FEATURES = {
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions_april",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
}
assert not (set(FEATURES) & FORBIDDEN_FEATURES)
X = df_data[FEATURES].copy()
y = df_data["is_declining_label"].astype(int).copy()
groups = df_data["client_hash_id"].copy()

SEED = 42
model_templates = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEED),
    ),
    "HistGradientBoosting": make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        HistGradientBoostingClassifier(
            max_iter=120,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=1.0,
            random_state=SEED,
        ),
    ),
}


def precision_at_k(labels, scores, k=50):
    label_array = np.asarray(labels)
    score_array = np.asarray(scores)
    take = min(k, len(label_array))
    order = np.argsort(-score_array, kind="mergesort")[:take]
    return float(label_array[order].mean())


position = df_data["gsc_avg_position_march"].fillna(0)
stale = df_data["content_age_days"].ge(180)
high_volume = df_data["gsc_impressions_march"].ge(500)
suboptimal_position = position.ge(4.0)
rule_scores = (
    stale.astype(int)
    * high_volume.astype(int)
    * suboptimal_position.astype(int)
    * df_data["gsc_impressions_march"]
    / (position + 1.0)
).to_numpy()

split_designs = {
    "Before: random rows": list(
        ShuffleSplit(n_splits=5, test_size=0.20, random_state=SEED).split(X, y)
    ),
    "After: client-held-out": list(
        GroupKFold(n_splits=5).split(X, y, groups=groups)
    ),
}
metric_rows = []
grouped_oof_scores = {
    name: np.full(len(df_data), np.nan) for name in model_templates
}
for design_name, splits in split_designs.items():
    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        if design_name == "After: client-held-out":
            assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
        y_test = y.iloc[test_idx]
        test_methods = {"Week-4 rule": rule_scores[test_idx]}
        for name, template in model_templates.items():
            fitted = clone(template)
            fitted.fit(X.iloc[train_idx], y.iloc[train_idx])
            scores = fitted.predict_proba(X.iloc[test_idx])[:, 1]
            test_methods[name] = scores
            if design_name == "After: client-held-out":
                grouped_oof_scores[name][test_idx] = scores
        for method, scores in test_methods.items():
            metric_rows.append(
                {
                    "split": design_name,
                    "fold": fold_number,
                    "method": method,
                    "test_rows": len(test_idx),
                    "base_rate": float(y_test.mean()),
                    "precision@50": precision_at_k(y_test, scores, 50),
                    "average_precision": average_precision_score(y_test, scores),
                }
            )

assert all(np.isfinite(scores).all() for scores in grouped_oof_scores.values())
metrics = pd.DataFrame(metric_rows)
summary = metrics.groupby(["split", "method"], sort=False).agg(
    folds=("fold", "nunique"),
    mean_test_base_rate=("base_rate", "mean"),
    mean_precision_at_50=("precision@50", "mean"),
    sd_precision_at_50=("precision@50", "std"),
    mean_average_precision=("average_precision", "mean"),
).reset_index()
print(f"Week-5 paired-month frame: {len(df_data):,} rows; {groups.nunique():,} clients.")
print(f"April impression-drop proxy base rate: {y.mean():.3f}.")
print(f"scikit-learn {sklearn.__version__}; grouped folds keep each client entirely in train or test.")
print("Precision@50 and average precision, mean +/- SD across folds (not confidence intervals):")
for row in summary.itertuples(index=False):
    print(
        f"{row.split} | {row.method}: base={row.mean_test_base_rate:.3f}; "
        f"P@50={row.mean_precision_at_50:.3f} +/- {row.sd_precision_at_50:.3f}; "
        f"AP={row.mean_average_precision:.3f}"
    )

model_grouped_p50 = summary.loc[
    (summary["split"] == "After: client-held-out")
    & summary["method"].isin(model_templates)
].set_index("method")["mean_precision_at_50"]
best_model_name = model_grouped_p50.idxmax()
print(f"Best grouped model by observed mean P@50: {best_model_name}.")
print("Random-row scores are an optimistic split diagnostic; client-grouped results are primary.")
print("The 0.5 label cutoff describes this next-month proxy, not an intervention outcome.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Week-5 paired-month frame: 143,206 rows; 45 clients.
April impression-drop proxy base rate: 0.517.
scikit-learn 1.6.1; grouped folds keep each client entirely in train or test.
Precision@50 and average precision, mean +/- SD across folds (not confidence intervals):
Before: random rows | Week-4 rule: base=0.517; P@50=0.372 +/- 0.079; AP=0.510
Before: random rows | Logistic Regression: base=0.517; P@50=0.640 +/- 0.075; AP=0.578
Before: random rows | HistGradientBoosting: base=0.517; P@50=0.964 +/- 0.038; AP=0.744
After: client-held-out | Week-4 rule: base=0.517; P@50=0.392 +/- 0.214; AP=0.512
After: client-held-out | Logistic Regression: base=0.517; P@50=0.688 +/- 0.157; AP=0.560
After: client-held-out | HistGradientBoosting: base=0.517; P@50=0.524 +/- 0.248; AP=0.560
Best grouped model by observed mean P@50: Logistic Regression.
Random-row scores are an optimistic split diagnostic; client-grouped results are primary.
The 0.5 label cutoff describes this next-month proxy, not an intervent

## 3. Leakage audit

The prediction boundary is March 31: each listed input is measured in March or is static content age as of March 31; April impressions define the label only. The audit below checks the final Week-5 feature list against direct targets, outcome-window values, identifiers, trend-derived fields, and product decision outputs. It also prints paired-month eligibility explicitly: requiring complete April coverage defines the observed population, but can select for measurement availability. I inspect anonymous out-of-fold mistakes without printing hashed IDs, URLs, or query text.

In [3]:
feature_timing = {
    "gsc_impressions_march": "March 1-31 measurement; before April label window",
    "gsc_clicks_march": "March 1-31 measurement; before April label window",
    "gsc_avg_position_march": "March 1-31 measurement; before April label window",
    "ga4_sessions_march": "March measurement; missing when GA4 was unavailable",
    "ga4_available_march": "March measurement-availability flag",
    "content_age_days": "Derived at March 31 from content creation date",
    "position_available_march": "Derived only from March position measurement",
}
leakage_findings = pd.DataFrame(
    [
        {"feature": feature, "timing / audit note": feature_timing[feature]}
        for feature in FEATURES
    ]
)
print("Final Week-5 feature timing audit:")
print(leakage_findings.to_string(index=False))
assert set(FEATURES) == set(feature_timing)
assert not (set(FEATURES) & FORBIDDEN_FEATURES)
assert "gsc_impressions_april" not in X.columns
assert "is_declining_label" not in X.columns
assert "client_hash_id" not in X.columns and "content_hash_id" not in X.columns
print("PASS: no IDs, April outcome, target/sibling fields, or product outputs in model inputs.")
print(
    "Population: March impressions >= 10 and paired March/April rows only; "
    "April coverage affects eligibility, so results are conditional on observed paired months."
)

scores = grouped_oof_scores[best_model_name]
actual = y.to_numpy()
predicted = (scores >= 0.5).astype(int)
false_positive = (predicted == 1) & (actual == 0)
false_negative = (predicted == 0) & (actual == 1)
print(
    f"Grouped out-of-fold mistakes at the illustrative 0.5 cutoff: "
    f"{false_positive.sum():,} false positives; {false_negative.sum():,} false negatives."
)
error_rows = []
for error_name, mask in [
    ("false positive", false_positive),
    ("false negative", false_negative),
]:
    candidates = np.flatnonzero(mask)
    if len(candidates):
        # Show a confident mistake, using only non-identifying measurement fields.
        index = candidates[np.argmax(np.abs(scores[candidates] - 0.5))]
        row = df_data.iloc[index]
        error_rows.append(
            {
                "error": error_name,
                "OOF score (not calibrated)": scores[index],
                "actual April proxy label": actual[index],
                "March impressions": row["gsc_impressions_march"],
                "March clicks": row["gsc_clicks_march"],
                "March position": row["gsc_avg_position_march"],
                "content age days": row["content_age_days"],
                "GA4 available": row["ga4_available_march"],
            }
        )
error_examples = pd.DataFrame(error_rows)
print("Anonymous held-out-client error examples (no row or client IDs):")
print(error_examples.to_string(index=False, float_format="%.3f"))

error_frame = pd.DataFrame(
    {
        "content_age_days": df_data["content_age_days"],
        "any_error": predicted != actual,
        "false_positive": false_positive,
        "false_negative": false_negative,
    }
)
error_frame["age_band"] = pd.cut(
    error_frame["content_age_days"],
    bins=[-np.inf, 180, 365, np.inf],
    labels=["under 180 days", "180-365 days", "over 365 days"],
)
print("Error rates by content-age band (descriptive):")
print(
    error_frame.groupby("age_band", observed=False)
    .agg(rows=("any_error", "size"), error_rate=("any_error", "mean"))
    .to_string(float_format="%.3f")
)

Final Week-5 feature timing audit:
                 feature                                 timing / audit note
   gsc_impressions_march   March 1-31 measurement; before April label window
        gsc_clicks_march   March 1-31 measurement; before April label window
  gsc_avg_position_march   March 1-31 measurement; before April label window
      ga4_sessions_march March measurement; missing when GA4 was unavailable
        content_age_days      Derived at March 31 from content creation date
     ga4_available_march                 March measurement-availability flag
position_available_march        Derived only from March position measurement
PASS: no IDs, April outcome, target/sibling fields, or product outputs in model inputs.
Population: March impressions >= 10 and paired March/April rows only; April coverage affects eligibility, so results are conditional on observed paired months.
Grouped out-of-fold mistakes at the illustrative 0.5 cutoff: 37,681 false positives; 35,145 false neg

## 4. Claim rewrite

The rewrite below is generated from the grouped-fold measurements above. It narrows the claim to the observed proxy label and eligible paired-month population. It does not say the model predicts recovery, that a refresh causes a change, or that this one-month evaluation guarantees performance on future clients or months.

In [4]:
grouped = summary[summary["split"] == "After: client-held-out"].set_index("method")
model_p50 = grouped.loc[best_model_name, "mean_precision_at_50"]
rule_p50 = grouped.loc["Week-4 rule", "mean_precision_at_50"]
grouped_base_rate = grouped.loc[best_model_name, "mean_test_base_rate"]
print("Overclaim to avoid: ‘The model predicts which pages will recover after a refresh.’")
print(
    "Evidence-matched wording: ‘In this observed March-to-April evaluation of "
    f"{len(df_data):,} eligible paired-month rows across {groups.nunique():,} clients, "
    f"the {best_model_name} measured mean precision@50 of {model_p50:.3f} "
    f"across five client-held-out folds, against a grouped test base rate of "
    f"{grouped_base_rate:.3f}; the unchanged Week-4 rule measured {rule_p50:.3f}. "
    "This is directional decision-support evidence for ordering review under this "
    "proxy and eligibility rule, not evidence that a refresh causes recovery or "
    "that performance will persist in future months.’"
)
print("The fold SD is descriptive variation; no confidence interval or causal effect is claimed.")

Overclaim to avoid: ‘The model predicts which pages will recover after a refresh.’
Evidence-matched wording: ‘In this observed March-to-April evaluation of 143,206 eligible paired-month rows across 45 clients, the Logistic Regression measured mean precision@50 of 0.688 across five client-held-out folds, against a grouped test base rate of 0.517; the unchanged Week-4 rule measured 0.392. This is directional decision-support evidence for ordering review under this proxy and eligibility rule, not evidence that a refresh causes recovery or that performance will persist in future months.’
The fold SD is descriptive variation; no confidence interval or causal effect is claimed.


## Self-check

- [x] All four audit sections include written reasoning and supporting code.
- [ ] The notebook runs top to bottom in Colab with the warehouse secret enabled.
- [x] Reported errors omit client/content IDs, URLs, and raw queries.
- [x] Claims use observed, measured, directional, and decision-support language; no causal claim is made.
- [ ] Executed notebook outputs are committed under `work/notebooks/`; submit the repository URL on the card.